# UniRig on Colab — rig a Tripo monster (non-humanoid OK)

Runs the real [VAST-AI UniRig](https://github.com/VAST-AI-Research/UniRig) on a cloud GPU — no humanoid lock-in, handles 2 arms + 2 legs + tail (+ jaw if it's separable geometry).

**Before running:** Runtime ▸ Change runtime type ▸ **GPU** (T4 is fine — cell 2b disables FlashAttention so any GPU works). Then **Runtime ▸ Run all**, and upload your FBX when cell 5 prompts.

Self-contained: builds a Python 3.11 venv (Colab is 3.12), patches the configs, rigs, and downloads a `_rigged.fbx`.

## 1. Confirm we have a GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > GPU, then rerun.'
print('OK — GPU:', torch.cuda.get_device_name(0))

## 2. Clone the UniRig repo

In [ ]:
%cd /content
![ -d UniRig ] || git clone https://github.com/VAST-AI-Research/UniRig.git
%cd /content/UniRig
!ls launch/inference

## 2b. Disable FlashAttention (so it runs on ANY GPU, incl. T4)
UniRig's configs request `flash_attention_2` (LLM) and `flash: True` (mesh encoder). FlashAttention needs Ampere+ and errors on Turing — this swaps to PyTorch SDPA, which runs on any GPU. Must say `Patched 1` or more.

In [ ]:
import glob, os, re
changed = []
for path in glob.glob('/content/UniRig/configs/**/*.yaml', recursive=True):
    with open(path) as f:
        txt = f.read()
    new = txt.replace('flash_attention_2', 'sdpa')          # LLM (skeleton)
    new = re.sub(r'flash:\s*True', 'flash: False', new)      # michelangelo encoder
    # PTv3 point-cloud encoder (skin) defaults enable_flash=True in code; inject the flag off.
    if 'enable_flash' not in new:
        new = re.sub(r'(?m)^(\s*)__target__:\s*ptv3obj\s*$',
                     lambda m: f"{m.group(0)}\n{m.group(1)}enable_flash: False", new)
    if new != txt:
        with open(path, 'w') as f:
            f.write(new)
        changed.append(path)
print(f'Patched {len(changed)} config file(s):')
for c in changed:
    print('  -', c)
assert changed, 'Patched 0 — no flash settings found; tell me and I will locate them.'

## 3. Build a Python 3.11 environment (Colab is 3.12 — UniRig needs 3.11)
`uv` grabs a standalone Python 3.11 and a matched **torch 2.4.0 + CUDA 12.1** stack. ~2–3 min, no kernel restart.

In [ ]:
# spconv-cu120 / bpy==4.2 / flash_attn only have 3.11 wheels, so we isolate a 3.11 venv at /content/uenv.
!pip install -q uv
!uv venv /content/uenv --python 3.11 --seed
!/content/uenv/bin/python --version
# Torch 2.4.0 + cu121 — the combo with matching PyG and flash-attn prebuilt wheels.
!uv pip install --python /content/uenv/bin/python -q torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121
!/content/uenv/bin/python -c "import torch; print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())"

## 4. Install UniRig dependencies (into the 3.11 venv)
Verbose on purpose — if any package fails, you'll see exactly which one.

In [ ]:
PY = '/content/uenv/bin/python'
# PyG extras matched to torch 2.4.0 + cu121
!uv pip install --python {PY} torch_scatter torch_cluster --find-links https://data.pyg.org/whl/torch-2.4.0+cu121.html
# spconv for CUDA 12.x
!uv pip install --python {PY} spconv-cu120
# Repo requirements EXCEPT flash_attn (prebuilt wheel in 4b). One-per-line so a failure is isolated & visible.
!uv pip install --python {PY} "transformers==4.51.3" "bpy==4.2" lightning pytorch_lightning python-box einops omegaconf addict timm fast-simplification trimesh open3d pyrender huggingface_hub wandb
# UniRig expects numpy 1.x
!uv pip install --python {PY} "numpy==1.26.4"
print('core deps installed')

## 4b. flash-attn (prebuilt wheel)
Still installed because UniRig imports it, but cell 2b means it won't actually be *used* at runtime. If the URL 404s, use the FALLBACK cell at the bottom.

In [ ]:
!uv pip install --python /content/uenv/bin/python https://github.com/Dao-AILab/flash-attention/releases/download/v2.6.3/flash_attn-2.6.3+cu123torch2.4cxx11abiFALSE-cp311-cp311-linux_x86_64.whl
!/content/uenv/bin/python -c "import flash_attn; print('flash_attn', flash_attn.__version__)"

## 4c. Put the venv on PATH + verify deps (DON'T SKIP)
Makes every `!` cell below use the 3.11 venv's `python` (the rig scripts call `python` internally) and confirms the key imports work.

In [ ]:
import os
os.environ['PATH'] = '/content/uenv/bin:' + os.environ['PATH']
os.environ['VIRTUAL_ENV'] = '/content/uenv'
with open('/content/_verify.py', 'w') as f:
    f.write('import bpy, lightning, spconv, torch_scatter, transformers\n')
    f.write("print('all imports OK | bpy', bpy.app.version_string)\n")
!which python
!python --version
!python /content/_verify.py

## 5. Upload your model
Pick your Tripo FBX (also accepts `.glb`, `.obj`, `.vrm`).

In [ ]:
%cd /content/UniRig
import os
from google.colab import files
os.makedirs('results', exist_ok=True)
up = files.upload()
MODEL = list(up.keys())[0]
NAME = os.path.splitext(MODEL)[0]
print('Uploaded:', MODEL, '| base name:', NAME)

## 6. Generate skeleton
First run downloads model weights from HuggingFace (once). Add `--seed 42` (any number) to nudge the skeleton.

*Confirmation it's patched: you should NOT see a `Flash Attention 2.0 ...` line or an `Ampere` error.*

In [ ]:
!bash launch/inference/generate_skeleton.sh --input {MODEL} --output results/{NAME}_skeleton.fbx

## 7. Generate skinning weights

In [ ]:
!bash launch/inference/generate_skin.sh --input results/{NAME}_skeleton.fbx --output results/{NAME}_skin.fbx

## 8. Merge into a final rigged FBX (Unity-native)
Want GLB instead? change both `.fbx` below to `.glb`.

In [ ]:
!bash launch/inference/merge.sh --source results/{NAME}_skin.fbx --target {MODEL} --output results/{NAME}_rigged.fbx
from google.colab import files
files.download(f'results/{NAME}_rigged.fbx')

---
### FALLBACK: flash-attn
Only if cell 4b failed. Compiles from source — slow (10–20 min), may need a High-RAM runtime.

In [ ]:
!uv pip install --python /content/uenv/bin/python flash-attn --no-build-isolation